# 🌿 CropVision AI: Ultimate Network-Proof Training

This is the final, most robust version of the training notebook.

### ✨ Features:
1. **Auto-Resume**: Loads weights AND the best accuracy score from Drive.
2. **Real-Time Save**: Saves only when it actually beats the previous best score.
3. **Corruption Shield**: Handles truncated images automatically.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import time
import copy
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "img_size": 300,
    "batch_size": 32,
    "lr": 5e-4,
    "epochs": 25,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_path": "/content/drive/MyDrive/best_model_vision_autosave.pth"
}

In [ ]:
!unzip -o -q /content/drive/MyDrive/dataset_final.zip -d /content/dataset

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(340),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.RandomResizedCrop(CONFIG["img_size"], scale=(0.7, 1.0)),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(CONFIG["img_size"] + 20),
    transforms.CenterCrop(CONFIG["img_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("/content/dataset/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("/content/dataset/val", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

num_classes = len(train_dataset.classes)

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs=20, device='cuda'):
    best_acc = 0.0
    
    # 🔄 AUTO-RESUME CHECK
    if os.path.exists(CONFIG["save_path"]):
        print(f"🔄 Found existing model on Drive. Loading to resume...")
        checkpoint = torch.load(CONFIG["save_path"], map_location=device)
        
        # Handle both old format (state_dict only) and new format (dict with acc)
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
            best_acc = checkpoint.get('acc', 0.0)
            print(f"✅ Resumed with previous Best Acc: {best_acc:.4f}")
        else:
            model.load_state_dict(checkpoint)
            print("✅ Resumed weights (Accuracy tracking will restart from 0 for safety).")
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # 🏆 SAVING WITH ACCURACY TRACKING
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                # Save as a dictionary so we can resume accuracy too!
                save_data = {
                    'state_dict': model.state_dict(),
                    'acc': best_acc
                }
                torch.save(save_data, CONFIG["save_path"])
                print(f'--🏆 NEW ALL-TIME BEST SAVED! Acc: {best_acc:.4f}')
        print()

    return model

y_train = train_dataset.targets
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.tensor(weights, dtype=torch.float).to(CONFIG["device"])

model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(CONFIG["device"])

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"])

model = train_model(model, {'train': train_loader, 'val': val_loader}, criterion, optimizer, num_epochs=CONFIG["epochs"], device=CONFIG["device"])
